# N021 · 集合、去重与连续序列

**目标：** 只从连续段起点扩展以避免重复工作。

**先修：** N014, N019。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 集合查询；起点判定；摊还扫描；重复数字。

**配套讲解来源：** [同名逐章意见](../../comment/021_hash_sets_sequence_dedup.md)。

本章阅读顺序：**先明白问题 → 补基础 → 手算 → 看核心片段 → 分段运行 → 核对测试**。需要复制使用时，跳到末尾“完整代码”，它包含全部依赖、实现和测试。

## 1. 通俗解释：输入什么，要得到什么

这一章解决一个看起来必须排序的问题：**最长连续序列**。给你一个无序整数数组，问其中的数字能拼出的最长"连续递增的一串"有多长。注意这里的"连续"指的是数值连续（1,2,3,4 这样首尾相接），不要求这些数字在原数组里挨着；同一个数字重复出现也只算一次。题目要求平均意义上快于排序的解法。

举个具体到数字的例子：输入 `[100,4,200,1,3,2]` → 输出 `4` → 为什么？数组里含有 1,2,3,4 这四个数值上连续的数（虽然它们在数组里东一个西一个），这是一条长度为 4 的连续链；其余的 100 和 200 各自孤零零成一条长度 1 的链，所以答案是 4。

本章还顺带一个迷你函数：**两个数组的去重交集**——返回同时出现在两个数组里的那些值，每个值只出现一次。例如 `[1,1,2]` 与 `[1,3]` 的去重交集是 `{1}`，因为只有 1 两边都有，且只列一次。

这道题最诱人的岔路是"先排序再扫一遍"，它能做对，但代价是 O(n log n)。本章的集合解能把总账压到平均 O(n)，代价只是多花一份存不同值的内存：

| 思路 | 核心动作 | 时间 | 空间 |
|---|---|---|---|
| 排序后顺序扫描 | 排好序，相邻差 1 就接长 | O(n log n) | O(u) |
| 集合 + 只从链首延伸 | 查前驱定链首，逐链走一遍 | 平均 O(n) | O(u) |

u 是不同值的个数，u ≤ n。题目点名要线性时间时，只有第二行达标。

## 2. 基础知识：先读懂这些词

- **集合（set）**：Python 的哈希集合结构，"每个元素只存一份、查询平均常数时间"。把数组倒进集合，一次就完成了去重，还为后面的极速"在不在"查询铺路。
- **连续链（consecutive chain）**：数值上一首接一尾的一串数，比如 1,2,3,4。一个有限的整数集合总能被唯一地切成若干条互不相交的极大链（谁也不属于别的更长链），比如 {100,4,200,1,3,2} 切成 [1,2,3,4]、[100]、[200] 三条。
- **前驱（predecessor）**：x 的前驱是 x−1。如果 x−1 也在集合里，x 就不是链的首；只有"x−1 不在集合里"的 x 才是某条链的链首。
- **起点判定（只在链首启动）**：这是本章的灵魂技巧——只对没有前驱的值发起向右延伸。要是从每个值都延伸一遍，同一条长链会被从头到尾反复扫很多遍；只从链首启动，每条链恰好被完整走一次。
- **摊还分析（amortized analysis）**：单看外层循环里某个 x，它可能触发一次很长的内层 while，看似很贵；但把所有内层延伸加起来，每个不同的值总共只被踏过一次，所以总账仍然便宜。"摊还"就是把偶发的大开销平摊到整个执行过程算总账的意思。
- **去重交集（unique intersection）**：两个序列共同拥有的值的集合，每个值至多出现一次。Python 里两个集合用 `&` 运算符直接得到交集。

## 3. 思路推导：从小例子开始

- **Step 1：先把数组变成集合。** `set(nums)` 一步完成两件事：去掉重复数字（重复对"最长链"毫无贡献），并把查询速度升级到平均常数时间。
- **Step 2：给"链"画像。** 答案一定来自某条极大连续链，而每条极大链有一个独一无二的特征——链首没有前驱。比如链 1,2,3,4 中，1 的前驱 0 不在集合里；而 2、3、4 的前驱都在，说明它们只是链的中段/尾部。
- **Step 3：只从链首启动延伸。** 遍历集合中的每个值 x，先问"x−1 在不在集合里"。在，说明 x 有人接管，跳过；不在，x 是链首，从这里开始不断问"x+1 在吗"，在就继续 +1，直到链条断裂。链条长度就是停下来的位置减去链首。
- **Step 4：记录最长。** 每走完一条链，与当前最优 best 比一比，谁大留谁。全部链走完，best 就是答案。
- **Step 5：算总账看为什么快。** 外层循环每个值做一次前驱检查共 O(u) 次（u 是不同值的个数）；内层的延伸虽然单次可能很长，但每条链只被自己的链首触发一次，所有内层步数加起来也恰是 O(u)。总账平均 O(n)（建集合是 O(n)，后续是 O(u)≤O(n)）。

**手算演示**（对应 “运行示例”部分 的表格，`values=set([100,4,200,1,3,2])`）：

| 候选值 x | x−1 在集合里吗 | 是否链首 | 若为链首：延伸过程 | 链长 |
|---|---|---|---|---|
| 1 | 0 不在 | 是 | 1→2→3→4→5 不在，停 | 4 |
| 2 | 1 在 | 否（跳过） | — | — |
| 3 | 2 在 | 否（跳过） | — | — |
| 4 | 3 在 | 否（跳过） | — | — |
| 100 | 99 不在 | 是 | 100→101 不在，停 | 1 |
| 200 | 199 不在 | 是 | 200→201 不在，停 | 1 |

三条链长度分别为 4、1、1，best=4，与函数返回值一致。特别注意 2、3、4 都被"x−1 在集合里"一句挡掉了——正是这一挡，避免了把链 1..4 反复扫四遍。

**数一数两种做法各自的步数**（就用上面这个 6 个元素的例子）：集合解的外层检查 6 次，内层延伸只发生在 1、100、200 三个链首上，共走 4+1+1=6 步，总步数 12；要是没有起点判定、对每个值都延伸，内层就要走 4+3+2+1+1+1=12 步，总步数 18。链越长这个差距越悬殊——链长 1..n 时，无判定版的内层总步数是 n+(n-1)+...+1 = n(n+1)/2，平方级。

**再来一个含重复数字的小演示**（`[1,1,2]`）：建集合后 values={1,2}，重复的第二个 1 直接消失。遍历到 1 时，0 不在集合里，1 是链首，向右走到 3 停下，链长 3−1=2；遍历到 2 时，1 在集合里，跳过。best=2，正是"两个 1 只算一个、链是 1,2"的答案。

## 4. 核心代码：longest_consecutive

先关注决定答案的状态与更新操作。这是完整实现的对应片段，不是另一套解法；导入、辅助结构及其余接口在下一节补全。

```python
def longest_consecutive(nums):
    values = set(nums)
    best = 0
    for x in values:
        if x - 1 not in values:
            y = x
            while y in values:
                y += 1
            best = max(best, y - x)
    return best
```

### 逐段读懂代码

### 函数 1：`longest_consecutive`（最长连续序列）

先看建集合与外层遍历。

```python
def longest_consecutive(nums):
    values=set(nums)
    best=0
    for x in values:
        if x-1 not in values:
```

- `values=set(nums)`：数组倒进集合，去重 + 换来平均常数时间的成员查询。后面所有"在不在"的判断都打给它。
- `best=0`：最优长度从 0 起步——这样空数组会自然返回 0（连一条链都没有），空输入不用单独写分支。
- `for x in values:`：遍历集合中每个**不同的**值（集合天然去重，重复数字只来一遍）。
- `if x-1 not in values:`：起点判定。前驱不在，x 才是一条链的头；否则 x 属于别人负责的链，直接跳过。这个 if 就是防止重复工作的闸门。

再看链首的向右延伸与记分。

```python
y=x
            while y in values: y+=1
            best=max(best,y-x)
    return best
```

- `y=x` 与 `while y in values: y+=1`：从链首出发向右试探。循环退出时 y 停在"第一个不在集合里的数"，也就是链尾再 +1 的位置。紧凑的 `while ...: y+=1` 把"检查 + 前进"压在一行。
- `best=max(best,y-x)`：链长恰好是 y−x。拿链 1..4 来说，y 最后停在 5，5−1=4 正是链长。谁大留谁，更新最优。
- `return best`：所有链首都试过了，返回冠军。

### 函数 2：`unique_intersection`（去重交集）

```python
def unique_intersection(a, b):
    return set(a) & set(b)
```

- 整个函数一行。`set(a)`、`set(b)` 分别把两个列表去重变成集合；`&` 是集合的交集运算，返回"两边都有的值"组成的新集合。
- 交集结果天然满足"每个元素至多一次"的去重契约，比如 `[1,1,2] & [1,3]` 得 `{1}`，左边的重复 1 不会让结果里出现两个 1。
- 这个小函数是集合"去重 + 快速成员判断"两大能力的浓缩展示，也是本章标题里"去重"二字的直接落点。

## 5. 分段实现：按顺序运行

**本章接口：** `longest_consecutive(nums)`、`unique_intersection(a, b)`

### longest_consecutive

In [1]:
def longest_consecutive(nums):
    values = set(nums)
    best = 0
    for x in values:
        if x - 1 not in values:
            y = x
            while y in values:
                y += 1
            best = max(best, y - x)
    return best

### unique_intersection

In [2]:
def unique_intersection(a, b):
    return set(a) & set(b)

## 6. 运行示例与状态可视化

先预测结果再运行。手算表帮助解释过程；下面的表由当前实现计算。若表只展示输入输出，它并不代替内部状态轨迹。

In [3]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

values=set([100,4,200,1,3,2]); rows=[]
for x in sorted(values):
    if x-1 not in values:
        chain=[]; y=x
        while y in values: chain.append(y); y+=1
        rows.append((x,chain,len(chain)))
show_table(['链首','延伸链','长度'],rows)

链首,延伸链,长度
1,"[1, 2, 3, 4]",4
100,[100],1
200,[200],1


## 7. 正确性、适用条件与复杂度

有限整数集合唯一分解为若干最大连续链，每条链有唯一无前驱起点；算法恰好遍历这些链并取最大长度。

**代价：** 平均 O(n) 时间、O(u) 空间；均摊分析需同时计外层起点检查与内部延伸。

### 展开理解

**为什么是对的：** 一个有限的整数集合可以这样看：把所有数按数值摆到数轴上，相邻的数自动粘成一串，整根链条在粘无可粘时停下来，每串就是一条"极大的连续链"。这样的切分是唯一的——每个数属于且只属于一条链。每条链恰好有一个链首（链上最小的那个数，它的前驱不在集合里），链上其他数的前驱都在集合里。我们的算法遍历所有值，但只对"前驱缺席"的链首发起延伸，而每个链首的延伸恰好把它那条链从头走到尾；链上非链首的值发起都被 if 挡掉。于是每条极大链被完整遍历恰好一次，链长计算 y−x 也精确无误，best 取的是所有链长的最大值——这正是"最长连续序列"的定义。重复数字在建集合那一步已经消掉，不会造成任何干扰。

**复杂度是多少：** 时间上，建集合是 O(n)；外层循环对每个不同值做一次"x−1 在不在"的检查，共 O(u) 次；内层 while 单独看某一次可能很长（比如链 1..100000 一次要走十万步），但所有链首的延伸步数加起来，恰好等于每个不同值被踏过的总次数，也就是 O(u)。这里必须"同时计外层检查与内层延伸"的总账（摊还视角），才能看出整体是平均 O(n)，而不是被单次长 while 吓到误判成平方。空间上，集合存了 u 个不同值，是 O(u)。举个具体的数：n=10⁵ 个数时，总操作量也就十万次上下的量级，一瞬间完成；对比"从每个元素都延伸"的错误做法，在 1..10⁵ 这条链上会做约 10¹⁰/2 次操作（从 1 延伸十万步、从 2 再延伸九万多步……），要等到天荒地老——这正是练习 2 要你亲手构造的反例。

**两个高频疑问，顺便说清：**

- 疑问一："外层 for 遍历 set，顺序是乱的，会不会漏链？"——不会。集合遍历顺序虽然乱，但每个值都会被轮到一次；而每条链的链首"前驱缺席"这个判定与顺序无关，谁先谁后都能把所有链走完，只是发现链的先后不同。
- 疑问二："重复数字会不会让 while 原地打转？"——不会。建集合时重复已经消掉，集合里每个值只有一份；while 是严格 +1 向右走的，访问过的位置不会再访问，不存在死循环。

## 8. 单元测试：每个用例在检查什么

（对应程序见下方分段实现与完整代码。）

- 第 1 条是**正常值测试**（也是 LeetCode 128 的官方样例）：乱序数组里藏着一根 1..4 的链，答案 4。它考察"数值连续但位置乱放"的核心场景。
- 第 2 条是**重复数字测试**（特殊值）：`[1,1,2]` 里的两个 1 只能算一个，链是 1,2，长度 2。这条钉住"去重后再算长度"的契约——不去重的话有人会数出 3。
- 第 3 条是**空输入测试**：空数组没有任何链，best 从 0 起步原样返回，考察空集合时循环一次不转的路径。
- 中间这段双重循环是**穷举差分测试**：枚举元素取 −1/0/1/3、长度 0~5 的全部数组。取值池是精心挑的：包含相邻值（−1,0,1）和断档值（1 与 3 之间缺 2），能造出各种"断链"形态。参照算法另起炉灶，分三步看：
  - `seq=sorted(set(a))`：先去重、再排序，把无序输入变成一列干净递增的值。
  - `run` 顺序扫描：遇到"x 恰好等于前一个 +1"就接长一段；紧凑表达式 `run+1 if prev is not None and x==prev+1 else 1` 的意思是——能接上就是 run+1，接不上（或这是第一个数）就重新从 1 开始。
  - `best` 随时取最大；扫完得到的 best 就是排序参照的答案。
  - 这是典型的 O(n log n) 排序参照，与被测的集合解思路完全不同，两者在每个输入上都必须给出相同答案。
- `assert unique_intersection([1,1,2],[1,3])=={1}`：**交集去重测试**。左边重复的 1 不影响结果，交集里 1 只出现一次，验证 `set & set` 的去重契约。
- `print(...)` 打印一句通过确认，方便肉眼检查这批穷举是否全部通过。

In [4]:
assert longest_consecutive([100, 4, 200, 1, 3, 2]) == 4

assert longest_consecutive([1, 1, 2]) == 2

assert longest_consecutive([]) == 0

from itertools import product

for n in range(6):
    for a in product([-1, 0, 1, 3], repeat=n):
        seq = sorted(set(a))
        best = run = 0
        prev = None
        for x in seq:
            run = run + 1 if prev is not None and x == prev + 1 else 1
            best = max(best, run)
            prev = x
        assert longest_consecutive(a) == best

assert unique_intersection([1, 1, 2], [1, 3]) == {1}

print('N021: 所有本章断言通过')

N021: 所有本章断言通过


## 9. 练习：先尝试，再看提示

**练习 1：** 返回最长连续段本身。

**练习 2：** 构造从每个元素扩展的平方级反例。

<details>
<summary>展开思路提示（不是完整答案）</summary>

- **练习 1（返回最长连续段本身）**：思路方向——现在的函数只算长度（y−x），你要把链上实际的数带出来。提示：在 while 延伸时顺便把 y 走过的值收集起来（或者只在确认这是新的最优链时，用"从链首 x 到 y−1"重建区间 `[x,y)`）；注意平局时的约定（两条链一样长时返回哪条？建议约定返回先遇到的那条，并写进契约）。手算例子用 `[100,4,200,1,3,2]`，期望返回 `[1,2,3,4]`。
- **练习 2（构造平方级反例）**：思路方向——把"没有起点判定"的错误版本写出来（对每个 x 都向右延伸并计长），然后喂给它一条长链。提示：取 `nums=list(range(1,n+1))`，比如 n=5 时数组是 `[1,2,3,4,5]`：错误版从 1 出发走 5 步、从 2 出发走 4 步、从 3 出发走 3 步……总共 5+4+3+2+1=15 步，是 n²/2 量级；对照正确版只从链首 1 出发走 5 步。写清输入、两种版本的步数对比，反例就成立了；n 取大一点（如 10⁴）时错误版会明显卡住，也是直观的证据。

两道练习验收时的共同要求：写清"输入、输出、边界"，并且能口头回答"为什么每个链首的 while 加起来只有 O(u) 步"——答得上来，摊还分析就算过关了。

</details>

## 10. 完整代码：可独立运行

下面这一格包含**全部导入、全部接口和验证用例**，不依赖前面的单元格。可以重启内核后只运行这一格，也可以复制到 `.py` 文件后运行。在 Notebook 中 Run All 时会再验证一次，这是为独立复制使用保留的完整版本。

In [5]:
# 完整实现：本单元格不依赖其他单元格。
def longest_consecutive(nums):
    values = set(nums)
    best = 0
    for x in values:
        if x - 1 not in values:
            y = x
            while y in values:
                y += 1
            best = max(best, y - x)
    return best

def unique_intersection(a, b):
    return set(a) & set(b)

# 示例与回归测试
assert longest_consecutive([100, 4, 200, 1, 3, 2]) == 4

assert longest_consecutive([1, 1, 2]) == 2

assert longest_consecutive([]) == 0

from itertools import product

for n in range(6):
    for a in product([-1, 0, 1, 3], repeat=n):
        seq = sorted(set(a))
        best = run = 0
        prev = None
        for x in seq:
            run = run + 1 if prev is not None and x == prev + 1 else 1
            best = max(best, run)
            prev = x
        assert longest_consecutive(a) == best

assert unique_intersection([1, 1, 2], [1, 3]) == {1}

print('N021: 所有本章断言通过')

N021: 所有本章断言通过


## 11. 代表题与迁移

- **128. Longest Consecutive Sequence（最长连续序列）**：这题就是 `longest_consecutive` 的原型——练的是"集合 + 链首判定 + 摊还 O(n)"这套组合拳，它要求线性时间，恰好逼你放弃排序走这条路。
- **349. Intersection of Two Arrays（两个数组的交集）**：这题就是 `unique_intersection` 的原型——练的是"集合去重 + 交集"的基本功，输出里每个元素只出现一次是它的关键契约。
- 迁移建议："只从等价类的代表（链首）启动工作"是个通用省时技巧——并查集按根合并、图遍历只从未访问点出发，背后都是同一个"别做重复工作"的思想。

**题面入口：** 

题目用于知识映射；本章完整代码遵循教学接口，不等于每道题的完整平台提交。

## 12. 掌握检查

合上代码，说明输入输出、核心变量的含义、每次更新为什么成立，以及一个边界情况。再完成一道练习，不能只凭“运行通过”判断掌握。

**继续学习：** [课程索引](../../COURSE_INDEX.md)。